# Phase 2 EDA: Static and Hybrid Defect Metrics

This notebook loads the PROMISE/NASA static metric CSVs, fuses repository-mined metrics when reliable join data exists, and saves exploratory plots to `outputs/eda/`. Repository metrics that are unavailable remain `NaN` and are explicitly excluded from hybrid EDA.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import (
    REPO_METRIC_COLUMNS,
    build_hybrid_features,
    load_and_clean,
)

sns.set_theme(style='whitegrid', context='notebook')
STATIC_DIR = PROJECT_ROOT / 'data' / 'static'
REPO_DIR = PROJECT_ROOT / 'data' / 'repo_mined'
EDA_DIR = PROJECT_ROOT / 'outputs' / 'eda'
EDA_DIR.mkdir(parents=True, exist_ok=True)
DATASETS = sorted(path.stem for path in STATIC_DIR.glob('*.csv'))
LABEL_COL = 'defects'
PROJECT_ROOT

## Load Static and Hybrid Datasets

In [ ]:
datasets = {}
notes = []

for name in DATASETS:
    static_path = STATIC_DIR / f'{name}.csv'
    repo_path = REPO_DIR / f'{name}_repo_metrics.csv'
    if not static_path.exists():
        notes.append(f'{name.upper()}: static CSV not found at {static_path}; run src/download_static_data.py when network access is available.')
        continue

    static_df = load_and_clean(static_path, label_col=LABEL_COL)
    repo_df = pd.read_csv(repo_path) if repo_path.exists() else pd.DataFrame()
    hybrid_df = build_hybrid_features(static_df, repo_df)
    match_rate = hybrid_df.attrs.get('hybrid_match_rate', 0.0)

    datasets[name] = {
        'static': static_df,
        'hybrid': hybrid_df,
        'hybrid_match_rate': match_rate,
        'hybrid_available': bool(hybrid_df['repo_metrics_available'].any()),
    }
    notes.append(f'{name.upper()}: static rows={len(static_df)}, hybrid repo match rate={match_rate:.2%}')

notes_path = EDA_DIR / 'eda_notes.md'
notes_path.write_text('\n'.join(f'- {note}' for note in notes), encoding='utf-8')
print('\n'.join(notes))

## Plot Helpers

In [ ]:
def numeric_feature_columns(df, exclude=()):
    exclude = set(exclude)
    return [col for col in df.select_dtypes(include=[np.number]).columns if col not in exclude]

def save_class_balance(df, dataset_name, variant):
    counts = df[LABEL_COL].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.barplot(x=counts.index.astype(str), y=counts.values, ax=ax, color='#4C78A8')
    ax.set_title(f'{dataset_name.upper()} {variant}: class balance')
    ax.set_xlabel('Defect label')
    ax.set_ylabel('Rows')
    fig.tight_layout()
    path = EDA_DIR / f'{dataset_name}_{variant}_class_balance.png'
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path

def save_corr_heatmap(df, dataset_name, variant, columns):
    columns = [col for col in columns if col in df.columns]
    if len(columns) < 2:
        return None
    corr = df[columns].corr(numeric_only=True)
    size = max(6, min(18, 0.45 * len(columns)))
    fig, ax = plt.subplots(figsize=(size, size))
    sns.heatmap(corr, cmap='vlag', center=0, square=False, ax=ax)
    ax.set_title(f'{dataset_name.upper()} {variant}: correlation heatmap')
    fig.tight_layout()
    path = EDA_DIR / f'{dataset_name}_{variant}_correlation_heatmap.png'
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path

def label_correlations(df, feature_columns):
    cols = [col for col in feature_columns if col in df.columns and col != LABEL_COL]
    if not cols:
        return pd.Series(dtype=float)
    corr = df[[*cols, LABEL_COL]].corr(numeric_only=True)[LABEL_COL].drop(labels=[LABEL_COL], errors='ignore')
    return corr.dropna().sort_values(key=lambda s: s.abs(), ascending=False)

## Static-Only EDA

In [ ]:
static_corr_rows = []

for name, bundle in datasets.items():
    df = bundle['static']
    static_features = numeric_feature_columns(df, exclude=[LABEL_COL])
    save_class_balance(df, name, 'static')
    save_corr_heatmap(df, name, 'static_metrics', [*static_features, LABEL_COL])
    corrs = label_correlations(df, static_features)
    for feature, corr in corrs.items():
        static_corr_rows.append({'dataset': name, 'feature_set': 'static', 'feature': feature, 'defect_correlation': corr})

static_corr_df = pd.DataFrame(static_corr_rows)
static_corr_df.to_csv(EDA_DIR / 'static_label_correlations.csv', index=False)
static_corr_df.head(20)

## Hybrid EDA Where Repository Metrics Are Available

Datasets with `repo_metrics_available == False` are skipped for hybrid plots. This avoids treating missing repository history as zero activity.

In [ ]:
comparison_rows = []
hybrid_skips = []

if not datasets:
    hybrid_skips.append('No static datasets were loaded; hybrid EDA is skipped until static CSVs are available.')

for name, bundle in datasets.items():
    hybrid = bundle['hybrid']
    if not bundle['hybrid_available']:
        hybrid_skips.append(f'{name.upper()}: skipped hybrid EDA because no rows have repository-mined metrics.')
        continue

    hybrid_available = hybrid[hybrid['repo_metrics_available']].copy()
    static_features = [col for col in numeric_feature_columns(bundle['static'], exclude=[LABEL_COL]) if col in hybrid_available.columns]
    repo_features = [col for col in REPO_METRIC_COLUMNS if col in hybrid_available.columns]

    save_class_balance(hybrid_available, name, 'hybrid')
    save_corr_heatmap(hybrid_available, name, 'repo_metrics', [*repo_features, LABEL_COL])
    save_corr_heatmap(hybrid_available, name, 'combined_metrics', [*static_features, *repo_features, LABEL_COL])

    static_corr = label_correlations(hybrid_available, static_features)
    repo_corr = label_correlations(hybrid_available, repo_features)
    static_best = static_corr.abs().max() if not static_corr.empty else np.nan
    repo_best = repo_corr.abs().max() if not repo_corr.empty else np.nan
    stronger = 'repo' if repo_best > static_best else 'static'
    comparison_rows.append({
        'dataset': name,
        'static_best_abs_corr': static_best,
        'repo_best_abs_corr': repo_best,
        'raw_correlation_stronger_feature_set': stronger,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(EDA_DIR / 'static_vs_repo_label_correlation_comparison.csv', index=False)
(EDA_DIR / 'hybrid_skip_notes.md').write_text('\n'.join(f'- {note}' for note in hybrid_skips), encoding='utf-8')
print('\n'.join(hybrid_skips) if hybrid_skips else 'Hybrid plots generated for all datasets with repository metrics.')
comparison_df

## Raw Correlation Interpretation

When hybrid repository metrics are available, compare `static_vs_repo_label_correlation_comparison.csv`: the feature set with the larger absolute correlation to the defect label is the stronger raw-correlation signal for that dataset. Correlation is only an initial screen; model-based evaluation, SHAP, and stability analysis should decide final usefulness.